# 02 - Experiment Tracking with MLflow on Azure ML

**Azure ML MLOps Workshop | Block 2**

### Why Experiment Tracking?

Without tracking, ML development is trial and error with no memory:
- Which hyperparameters gave the best F1 score?
- Did TF-IDF with bigrams beat unigrams?
- What was the training data version?

Azure ML uses **MLflow** as its native tracking layer. Every run automatically captures
parameters, metrics, and artifacts - queryable and comparable in Studio.

### What you'll do:
1. Set up MLflow tracking to Azure ML
2. Train multiple classifiers with autologging
3. Log custom metrics and artifacts
4. Compare runs in Azure ML Studio

---

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<YOUR_SUBSCRIPTION_ID>"  # <<<< CHANGE THIS TO YOUR AZURE SUBSCRIPTION ID
RESOURCE_GROUP = "<YOUR_RESOURCE_GROUP>"  # <<<< CHANGE THIS TO YOUR RESOURCE GROUP (e.g., rg-aml-workshop-jd)
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"  # <<<< CHANGE THIS TO YOUR WORKSPACE NAME (e.g., aml-workshop-jd)

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
print(f"Connected to: {ml_client.workspace_name}")

## Step 1: Configure MLflow Tracking

Azure ML workspaces have a built-in MLflow tracking server. We just point MLflow at it.

In [ ]:
import mlflow

workspace = ml_client.workspaces.get(WORKSPACE_NAME)
tracking_uri = workspace.mlflow_tracking_uri
mlflow.set_tracking_uri(tracking_uri)
print(f"MLflow tracking URI: {tracking_uri}")

EXPERIMENT_NAME = "contoso-lead-classifier"
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Experiment: {EXPERIMENT_NAME}")

## Step 2: Load and Prepare Data

We'll use the cleaned dataset (v2) that we registered in the previous notebook.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../src/track_a_text"))

import pandas as pd
from preprocess import load_and_clean_inspections, prepare_train_test

df = load_and_clean_inspections("../../data/inspections_dataset.csv")
print(f"Dataset: {len(df):,} rows")
print(f"Label distribution: {df['label'].value_counts().to_dict()}")

In [ ]:
X_train, X_test, y_train, y_test, vectorizer, train_df, test_df = prepare_train_test(
    df, test_size=0.2, max_features=5000
)

print(f"Train: {X_train.shape[0]:,} samples, {X_train.shape[1]:,} features")
print(f"Test:  {X_test.shape[0]:,} samples")
print(f"Train label dist: {pd.Series(y_train).value_counts().to_dict()}")

## Step 3: Train Multiple Models with MLflow Tracking

We'll train three classifiers and track everything automatically.
Each run captures: parameters, metrics, the model artifact, and custom plots.

### Run 1: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import numpy as np
import joblib

def train_and_log(model, model_name, X_train, X_test, y_train, y_test, vectorizer, data_version="2"):
    """Train a model and log everything to MLflow."""
    with mlflow.start_run(run_name=model_name) as run:
        mlflow.log_param("data_asset", "classified-inspections")
        mlflow.log_param("data_version", data_version)
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("n_train", X_train.shape[0])
        mlflow.log_param("n_test", X_test.shape[0])
        mlflow.log_param("n_features", X_train.shape[1])

        X_tr = X_train.toarray() if hasattr(X_train, 'toarray') and 'Boosting' in model_name else X_train
        X_te = X_test.toarray() if hasattr(X_test, 'toarray') and 'Boosting' in model_name else X_test

        model.fit(X_tr, y_train)
        y_pred = model.predict(X_te)
        y_proba = model.predict_proba(X_te)[:, 1]

        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred),
        }
        mlflow.log_metrics(metrics)

        report = classification_report(y_test, y_pred, target_names=["No Lead", "Lead"])
        report_path = f"{model_name}_report.txt"
        with open(report_path, "w") as f:
            f.write(report)
        mlflow.log_artifact(report_path)

        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay.from_predictions(
            y_test, y_pred, display_labels=["No Lead", "Lead"], ax=ax, cmap="Blues"
        )
        ax.set_title(f"{model_name} - Confusion Matrix")
        cm_path = f"{model_name}_confusion_matrix.png"
        fig.savefig(cm_path, dpi=100, bbox_inches="tight")
        mlflow.log_artifact(cm_path)
        plt.show()

        mlflow.sklearn.log_model(model, artifact_path="model")

        vectorizer_path = "vectorizer.joblib"
        joblib.dump(vectorizer, vectorizer_path)
        mlflow.log_artifact(vectorizer_path, artifact_path="model")

        os.remove(report_path)
        os.remove(cm_path)
        os.remove(vectorizer_path)

        print(f"\n{model_name} results:")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}")
        print(f"  Run ID: {run.info.run_id}")

        return run.info.run_id, metrics

In [ ]:
lr_run_id, lr_metrics = train_and_log(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42, C=1.0),
    "LogisticRegression",
    X_train, X_test, y_train, y_test,
    vectorizer=vectorizer,
)

### Run 2: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_run_id, rf_metrics = train_and_log(
    RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1),
    "RandomForest",
    X_train, X_test, y_train, y_test,
    vectorizer=vectorizer,
)

### Run 3: Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_run_id, gb_metrics = train_and_log(
    GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42),
    "GradientBoosting",
    X_train, X_test, y_train, y_test,
    vectorizer=vectorizer,
)

### Run 4: Logistic Regression with Different Regularization

Demonstrating how experiment tracking captures hyperparameter variations.

In [ ]:
lr2_run_id, lr2_metrics = train_and_log(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42, C=0.1),
    "LogisticRegression_C0.1",
    X_train, X_test, y_train, y_test,
    vectorizer=vectorizer,
)

## Step 4: Compare Runs

### Programmatic comparison

In [ ]:
all_results = {
    "LogisticRegression": lr_metrics,
    "RandomForest": rf_metrics,
    "GradientBoosting": gb_metrics,
    "LogisticRegression_C0.1": lr2_metrics,
}

results_df = pd.DataFrame(all_results).T
results_df = results_df.sort_values("f1", ascending=False)
print("\nModel Comparison (sorted by F1):")
print(results_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, metric in enumerate(["accuracy", "precision", "recall", "f1"]):
    results_df[metric].plot(kind="barh", ax=axes[i], color=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"])
    axes[i].set_title(metric.capitalize())
    axes[i].set_xlim(0, 1)
plt.suptitle("Contoso Lead Classifier - Model Comparison", fontsize=14)
plt.tight_layout()
plt.show()

### Query runs via MLflow API

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    max_results=20,
)

finished_runs = [r for r in runs if r.info.status == "FINISHED" and "f1" in r.data.metrics]
finished_runs.sort(key=lambda r: r.data.metrics.get("f1", 0), reverse=True)

print("Top runs by F1 score:")
for run in finished_runs[:5]:
    print(f"  {run.data.params.get('model_type', 'N/A'):30s} | "
          f"F1: {run.data.metrics.get('f1', 0):.4f} | "
          f"Run: {run.info.run_id[:8]}")

## Go to Azure ML Studio Now

Navigate to **Jobs > contoso-lead-classifier** and explore:

1. **Run list** - See all 4 runs with their metrics
2. **Select 2+ runs > Compare** - Use the comparison view
3. **Click a run > Metrics** - See logged metrics
4. **Click a run > Outputs + Logs** - Find the confusion matrix PNG and classification report
5. **Click a run > Parameters** - See the data version and model config

This is the visual payoff of MLflow tracking.

---

## Key Takeaways

| Concept | What We Did |
|---------|------------|
| **Autologging** | MLflow automatically captured model params |
| **Custom metrics** | We logged accuracy, precision, recall, F1 |
| **Artifacts** | Confusion matrices and reports stored with each run |
| **Data lineage** | Every run records which data version it used |
| **Comparison** | Both programmatic and Studio UI comparison |

---

**Next:** [03 - Model Registration](./03_model_registration.ipynb)